# OCR V6.1 trên Google Colab GPU

## Mục tiêu
Chạy VietOCR tiếng Việt cùng các đầu ra V6 từ [VuTuanAn1403/OCR](https://github.com/VuTuanAn1403/OCR), lưu PDF/model/kết quả trên Google Drive.
Chọn **Runtime → Change runtime type → GPU**, rồi chạy các ô từ trên xuống.

- `balanced`: VietOCR + Docling TableFormer/CodeFormula cho máy lab (mặc định).
- `local`: VietOCR dùng GPU, bảng heuristic/vector, không cài Docling.
- Notebook cài Python 3.11 riêng; không thay các gói OCR vào Python của kernel Colab.
- Kiểm tra cấu trúc/cú pháp và giải dependency Linux đã thực hiện; **chưa chạy trực tiếp trên Colab**.
- GPU/giới hạn phiên thay đổi theo Colab. Số đo GPU không thay nghiệm thu CPU hay độ chính xác trên ground truth.

## 1. Cấu hình
Giữ `balanced` để dùng bảng/công thức. Có thể đổi `REVISION` từ `main` sang commit SHA để tái lập.
Sau khi chạy thử một trang thành công, đặt `RUN_FULL_BATCH = True` và chạy lại ô xử lý lô.

In [ ]:
import datetime
import json
import os
from pathlib import Path
import shutil
import signal
import subprocess
import sys

REPO_URL = "https://github.com/VuTuanAn1403/OCR.git"
REVISION = "main"
PROFILE = "balanced"                 # "balanced", "accuracy", "local"
DRIVE_ROOT = Path("/content/drive/MyDrive/ocr_pdf")
REPO_DIR = Path("/content/OCR")
RUN_FULL_BATCH = False
FILE_LIMIT = None                    # Ví dụ 2; None xử lý mọi PDF
SHARD, SHARDS = 0, 1                  # Một phiên: 0, 1. Chỉ đổi khi chủ động chia lô.
ASSETS = "none"                      # "jpeg" để lưu thêm ảnh trích xuất
OUTPUT_MODE = "compact"              # Giữ đủ pages/ và bốn artifact V6
assert PROFILE in {"local", "balanced", "accuracy"}
assert SHARDS >= 1 and 0 <= SHARD < SHARDS
assert FILE_LIMIT is None or (type(FILE_LIMIT) is int and FILE_LIMIT > 0)
if not shutil.which("nvidia-smi"):
    raise RuntimeError("Hãy chọn Runtime → Change runtime type → GPU rồi chạy lại.")
subprocess.run(["nvidia-smi"], check=True)


## 2. Google Drive
Up PDF vào **MyDrive/ocr_pdf/data/raw/** (đuôi `.pdf`, trực tiếp trong thư mục này).
Model, cache và kết quả nằm trên Drive; môi trường Python và mã nguồn nằm trên ổ tạm của Colab.
Không ghi token/mật khẩu vào notebook.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
for folder in ("data/raw", "outputs", "cache", "models", "logs"):
    (DRIVE_ROOT / folder).mkdir(parents=True, exist_ok=True)
INPUT_DIR = DRIVE_ROOT / "data/raw"
BATCH_OUTPUT = DRIVE_ROOT / "outputs" / PROFILE
os.environ.update({
    "OCR_DATA_DIR": str(INPUT_DIR),
    "OCR_OUTPUT_DIR": str(BATCH_OUTPUT),
    "OCR_CACHE_DIR": str(DRIVE_ROOT / "cache" / PROFILE),
    "OCR_MODELS_DIR": str(DRIVE_ROOT / "models"),
    "OCR_DOCLING_PYTHON": str(REPO_DIR / ".venv-docling/bin/python"),
    "PYTHONUNBUFFERED": "1",
})
print("PDF:", INPUT_DIR)
print("Kết quả:", BATCH_OUTPUT)


## 3. Cài đặt
### Mã nguồn và Python 3.11
Lấy mã từ GitHub, ghi lại SHA thực chạy. Các lệnh dừng ngay khi lỗi; không tự xóa thay đổi trong checkout đã có.

In [ ]:
if not (REPO_DIR / ".git").exists():
    if REPO_DIR.exists():
        raise RuntimeError(f"Thư mục {REPO_DIR} đã tồn tại nhưng không phải repo. Chọn REPO_DIR khác.")
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_DIR)], check=True)
remote = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True).strip()
assert remote.removesuffix(".git") == REPO_URL.removesuffix(".git"), remote
subprocess.run(["git", "fetch", "--depth", "1", "origin", REVISION], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
COMMIT_SHA = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
print("Commit:", COMMIT_SHA)
subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.12.10"], check=True)
subprocess.run(["uv", "python", "install", "3.11"], check=True)
PYTHON_311 = subprocess.check_output(["uv", "python", "find", "3.11"], text=True).strip()
print("Python cài OCR:", PYTHON_311)


### VietOCR CUDA và Docling cách ly
Cài wheel PyTorch CUDA 12.6 chính thức, xác nhận GPU ở cả hai môi trường khi dùng Docling.
Lần đầu cần tải thư viện/model. Trọng số VietOCR đã lưu sẽ được xác minh và dùng lại.

In [ ]:
def run_logged(command, log_path):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", encoding="utf-8") as log:
        log.write("\n" + str(command) + "\n")
        process = subprocess.Popen(command, cwd=REPO_DIR, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                                   errors="replace", bufsize=1, start_new_session=True)
        try:
            for line in process.stdout:
                print(line, end="")
                log.write(line)
                log.flush()
            status = process.wait()
        except BaseException:
            # Stop the CLI and its Docling worker together on interruption.
            try:
                os.killpg(process.pid, signal.SIGTERM)
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()
            except ProcessLookupError:
                pass
            raise
        if status:
            raise subprocess.CalledProcessError(status, command)

setup = ["bash", "scripts/setup.sh", "--device", "cuda", "--python", PYTHON_311,
         "--models-dir", str(DRIVE_ROOT / "models")]
if PROFILE == "local":
    setup.append("--no-docling")
run_logged(setup, DRIVE_ROOT / "logs/setup.log")
OCR_PYTHON = str(REPO_DIR / ".venv/bin/python")
(DRIVE_ROOT / "logs/source_commit.txt").write_text(COMMIT_SHA + "\n", encoding="utf-8")
for environment in (".venv", ".venv-docling"):
    freeze = REPO_DIR / environment / "installed-requirements.txt"
    if freeze.exists():
        shutil.copyfile(freeze, DRIVE_ROOT / "logs" / f"{environment[1:]}-{COMMIT_SHA[:12]}.txt")


## 4. Kiểm tra GPU và chạy thử một trang
Xác nhận model VietOCR thực sự ở `cuda:0`. Với `balanced`/`accuracy`, chạy làm nóng Docling để tải model bảng/công thức;
lần đầu có thể mất nhiều thời gian hơn các trang tiếp theo.

In [ ]:
gpu_probe = """
import json, sys, torch
from src.ocr.application.convert_document import ConvertDocumentUseCase
from src.ocr.infrastructure.ocr.vietnamese_seq2seq_engine import VietnameseSeq2SeqEngine
assert torch.cuda.is_available(), "CUDA unavailable"
config = ConvertDocumentUseCase(thread_budget=2).resolve_profile(sys.argv[1], "cuda")
torch.set_num_threads(2)
engine = VietnameseSeq2SeqEngine(batch_size=config["batch_size"], device="cuda", num_threads=2)
actual = str(next(engine._recognizer.model.parameters()).device)
assert actual.startswith("cuda"), actual
print(json.dumps({"torch": torch.__version__, "cuda": torch.version.cuda,
                  "gpu": torch.cuda.get_device_name(0), "vietocr_model": actual,
                  "batch_size": config["batch_size"], "profile": sys.argv[1]}, indent=2))
"""
import textwrap
run_logged([OCR_PYTHON, "-c", textwrap.dedent(gpu_probe), PROFILE], DRIVE_ROOT / "logs/gpu_check.log")
pdf_files = sorted(INPUT_DIR.glob("*.pdf"))
if not pdf_files:
    raise FileNotFoundError(f"Up PDF vào {INPUT_DIR}, rồi chạy lại ô này.")
SAMPLE_PDF = pdf_files[0]
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
SMOKE_OUTPUT = DRIVE_ROOT / "outputs" / f"smoke-{PROFILE}-{stamp}"
smoke = [OCR_PYTHON, "run_ocr.py", str(SAMPLE_PDF), "--hardware", "colab_gpu",
         "--profile", PROFILE, "--pages", "1", "--device", "cuda", "--benchmark",
         "--assets", ASSETS, "--output-mode", OUTPUT_MODE, "--output", str(SMOKE_OUTPUT)]
if PROFILE != "local":
    smoke.append("--warmup-docling")
run_logged(smoke, DRIVE_ROOT / "logs" / f"smoke-{stamp}.log")
artifact_dir = SMOKE_OUTPUT / SAMPLE_PDF.stem
required = ("extraction.json", "structure.json", "document.json", "document.md", "COMPLETED.json")
assert all((artifact_dir / name).is_file() for name in required), artifact_dir
validation = json.loads((artifact_dir / "validation.json").read_text(encoding="utf-8"))
print("Validation:", validation["status"], "— NEEDS_REVIEW cần kiểm tra nội dung.")
print("Đầu ra thử:", artifact_dir)


## 5. Chạy lô
Đặt `RUN_FULL_BATCH = True` trong ô cấu hình để chạy toàn bộ, hoặc đặt `FILE_LIMIT` để giới hạn số PDF.
Nếu phiên bị ngắt, chạy lại ô này sau khi thiết lập lại môi trường: tài liệu đã hoàn tất hợp lệ được bỏ qua;
cache còn tương thích có thể dùng để tiếp tục. Mặc định một worker, một shard bao phủ toàn bộ đầu vào.

In [ ]:
if RUN_FULL_BATCH:
    batch = [OCR_PYTHON, "run_batch.py", "--hardware", "colab_gpu", "--profile", PROFILE,
             "--input", str(INPUT_DIR), "--output", str(BATCH_OUTPUT), "--device", "cuda",
             "--cache-dir", os.environ["OCR_CACHE_DIR"], "--workers", "1", "--threads-per-worker", "2",
             "--assets", ASSETS, "--output-mode", OUTPUT_MODE,
             "--excel", str(BATCH_OUTPUT / "OCR_BENCHMARK_RESULTS.xlsx")]
    if FILE_LIMIT is not None:
        batch.extend(["--limit", str(FILE_LIMIT)])
    if SHARDS > 1:
        batch.extend(["--shard", f"{SHARD}/{SHARDS}"])
    run_logged(batch, DRIVE_ROOT / "logs" / f"batch-{PROFILE}-{SHARD}-of-{SHARDS}.log")
    print("Kết quả:", BATCH_OUTPUT)
else:
    print("Đã chuẩn bị xong. Đặt RUN_FULL_BATCH = True ở ô cấu hình rồi chạy lại ô xử lý lô.")


## 6. Kiểm tra kết quả và bước tiếp theo
Các file V6 nằm trong thư mục từng tài liệu. `NEEDS_REVIEW` là cảnh báo nội dung, không phải xác nhận độ chính xác.
Khi chia shard, chỉ gộp sau khi mọi shard hoàn tất. Giữ nguyên thư mục `pages/` khi sao chép kết quả compact.

In [ ]:
from itertools import islice
for report_path in islice(BATCH_OUTPUT.rglob("batch_report.json"), 5):
    report = json.loads(report_path.read_text(encoding="utf-8"))
    print(report_path)
    print(json.dumps(report.get("summary", {}), ensure_ascii=False, indent=2))
print("PDF:", INPUT_DIR)
print("Model:", DRIVE_ROOT / "models")
print("Báo cáo và kết quả:", BATCH_OUTPUT)
print("Log và phiên bản môi trường:", DRIVE_ROOT / "logs")


### Gộp shard (chỉ chạy khi đã hoàn tất mọi shard)
Bỏ qua ô này nếu dùng mặc định `SHARDS = 1`.

In [ ]:
MERGE_COMPLETED_SHARDS = False
if MERGE_COMPLETED_SHARDS:
    assert SHARDS > 1, "Không cần gộp khi chỉ có một shard."
    run_logged([OCR_PYTHON, "run_batch.py", "--output", str(BATCH_OUTPUT), "--merge-shards", str(SHARDS)],
               DRIVE_ROOT / "logs/merge-shards.log")
